### In this notebook we run triton inference server with pretrained pytorch model and xgboost model. The pytorch model is used to generate future customer profiles as features. The xgboost model is used to predict whether or not the customer will default in future. 

In [1]:
import os

In [2]:
from rnn import TestRnnDataset,load_yaml
from torch.utils.data import DataLoader

In [3]:
import cudf
import numpy as np
import gc

import time
import tritonclient.grpc as triton_grpc
import tritonclient.http as httpclient
from tritonclient.utils import triton_to_np_dtype
from tritonclient import utils as triton_utils
HOST = 'localhost'
PORT = 8001
TIMEOUT = 60

In [4]:
PATH = 'data/amex'

# Data Preprocessing

In [5]:
%%time

train = cudf.read_parquet(f'{PATH}/train.parquet')
train['cid'], _ = train.customer_ID.factorize()
train['S_2'] = cudf.to_datetime(train['S_2'])

mask = train['cid']%4 == 0
test = train.loc[mask]
test = test.sort_values(['cid','S_2'])
test = test.reset_index(drop=True)
del train
gc.collect()


CPU times: user 632 ms, sys: 746 ms, total: 1.38 s
Wall time: 1.03 s


0

In [6]:
config = load_yaml('rnn.yaml')

Config(model='rnn', epochs=5, batch_size=512, seq=5, H1=512, H2=128, layers=1, E=192, dropout=0, lr=0.001, wd=0.0, tcols='all')


In [7]:
test_ds = TestRnnDataset(test,config)

RnnDataset not used columns:
['customer_ID', 'cid', 'S_2']


In [8]:
batch_size = config.batch_size
cpu_workers = 4

test_dl = DataLoader(test_ds, batch_size=batch_size,
                    shuffle=False, num_workers=cpu_workers,
                    drop_last=False)

# Launch Triton Server

In [9]:
x,xo = next(iter(test_dl))

In [10]:
x.shape,x.dtype,xo.shape,xo.dtype

(torch.Size([512, 5, 177]),
 torch.float32,
 torch.Size([512, 5, 177]),
 torch.float32)

In [11]:
TRITON_IMAGE = 'nvcr.io/nvidia/tritonserver:25.10-py3'

In [12]:
!docker pull {TRITON_IMAGE}

25.10-py3: Pulling from nvidia/tritonserver
Digest: sha256:9ff4dd7a1c52487b35e98208ce9bbdb6580b036cdc8943a64587e1c2685de662
Status: Image is up to date for nvcr.io/nvidia/tritonserver:25.10-py3
nvcr.io/nvidia/tritonserver:25.10-py3


In [13]:
cwd = os.getcwd()
cmd = f"docker run -p 8000:8000 -p 8001:8001 --gpus device=0 \
  -v {cwd}/model_repository:/models \
  {TRITON_IMAGE} \
  bash -c \"pip install xgboost scikit-learn && tritonserver --model-repository=/models --exit-on-error=false\" &"
cmd

'docker run -p 8000:8000 -p 8001:8001 --gpus device=0   -v /home/drollins/triton_amex_blackwell/model_repository:/models   nvcr.io/nvidia/tritonserver:25.10-py3   bash -c "pip install xgboost scikit-learn && tritonserver --model-repository=/models --exit-on-error=false" &'

In [16]:
os.system(cmd)

0


== Triton Inference Server ==

NVIDIA Release 25.10 (build 226999189)
Triton Server Version 2.62.0

Copyright (c) 2018-2025, NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

GOVERNING TERMS: The software and materials are governed by the NVIDIA Software License Agreement
(found at https://www.nvidia.com/en-us/agreements/enterprise-software/nvidia-software-license-agreement/)
and the Product-Specific Terms for NVIDIA AI Products
(found at https://www.nvidia.com/en-us/agreements/enterprise-software/product-specific-terms-for-ai-products/).

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

I1106 17:29:11.718041 1 pinned_memory_manager.cc:277] "Pinned memory pool is created at '0x7f9672000000' with size 268435456"
I1106 17:29:11.718083 1 cuda_memory_manager.cc:107] "CUDA memory pool is created on device 0 with size 67108864"
I1106 17:29:11.718869 1 model_lifecycle.cc:473] "loading: AutoRegressiveRNN:1"
I1106 17:29:11.718882 1 model_lifecycle.cc:473] "loading: amex_xgb:1"
I1106 17:29:11.908368 1 libtorch.cc:2509] "TRITONBACKEND_Initialize: pytorch"
I1106 17:29:11.908383 1 libtorch.cc:2519] "Triton TRITONBACKEND API version: 1.19"
I1106 17:29:11.908384 1 libtorch.cc:2525] "'pytorch' TRITONBACKEND API version: 1.19"
I1106 17:29:11.908401 1 libtorch.cc:2558] "TRITONBACKEND_ModelInitialize: AutoRegressiveRNN (version 1)"
I1106 17:29:11.908959 1 libtorch.cc:358] "Optimized execution is enabled for model instance 'AutoRegressiveRNN'"
I1106 17:29:11.908962 1 libtorch.cc:377] "Cache Cleaning is disabled for model instance 'AutoRegressiveRNN'"
I1106 17:29:11.908964 1 libtorch.cc:39

W1106 17:29:11.908725 1 libtorch.cc:329] "skipping model configuration auto-complete for 'AutoRegressiveRNN': not supported for pytorch backend"


I1106 17:29:11.932725 1 model_lifecycle.cc:849] "successfully loaded 'AutoRegressiveRNN'"
I1106 17:29:13.478112 1 python_be.cc:2289] "TRITONBACKEND_ModelInstanceInitialize: amex_xgb_0_0 (GPU device 0)"


/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py:1124: UserWarning: [17:29:13] WARNING: /workspace/src/c_api/c_api.cc:1511: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)


I1106 17:29:13.992907 1 model_lifecycle.cc:849] "successfully loaded 'amex_xgb'"
I1106 17:29:13.993071 1 server.cc:611] 
+------------------+------+
| Repository Agent | Path |
+------------------+------+
+------------------+------+

I1106 17:29:13.993102 1 server.cc:638] 
+---------+---------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
| Backend | Path                                                    | Config                                                                                                                                                        |
+---------+---------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
| pytorch | /opt/tritonserver/backen

### Initiate triton client

In [17]:
client = triton_grpc.InferenceServerClient(url=f'{HOST}:{PORT}')

In [18]:
def triton_predict(model_name, arr):
    triton_input = triton_grpc.InferInput('input__0', arr.shape, 'FP32')
    triton_input.set_data_from_numpy(arr)
    triton_output = triton_grpc.InferRequestedOutput('output__0')
    response = client.infer(model_name, model_version='1', inputs=[triton_input], outputs=[triton_output])
    return response.as_numpy('output__0')

In [19]:
rnn_fea = triton_predict('AutoRegressiveRNN',x.numpy())
rnn_fea.shape

(512, 13, 177)

In [20]:
x = np.hstack([xo[:,-1,:],rnn_fea[:,-1,:]])
x.shape

(512, 354)

In [21]:
pred = triton_predict('amex_xgb',x)
pred.shape

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [17:30:29] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


(512, 2)

# Run triton inference on validation data

In [22]:
%%time

yps = []
for x,xo in test_dl:
    rnn_fea = triton_predict('AutoRegressiveRNN',x.numpy())
    x = np.hstack([xo[:,-1,:],rnn_fea[:,-1,:]])
    pred = triton_predict('amex_xgb',x)
    yps.append(pred)
yp = np.vstack(yps)

CPU times: user 950 ms, sys: 474 ms, total: 1.42 s
Wall time: 7.19 s


In [23]:
yp.shape

(114729, 2)

In [24]:
%%time
test = test.drop_duplicates('cid')
trainl = cudf.read_csv(f'{PATH}/train_labels.csv')
test = test.merge(trainl, on='customer_ID', how='left')
test = test.sort_values('cid')
test.head()

CPU times: user 31.3 ms, sys: 24.3 ms, total: 55.7 ms
Wall time: 48.6 ms


,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,...,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145,cid,target
1696,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-03-09,0.938469,0,0.008724,1.006838,0.009228,0.124035157,0.0,0.004709,...,-1,0,0,0.000000,<NA>,0,0.000610,0,0,0
1697,00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8a...,2017-03-30,0.936842,0,0.003433,0.818691,0.007243,0.166190118,0.0,0.005927,...,-1,0,0,0.000000,<NA>,0,0.003867,0,4,0
1698,0000f99513770170a1aba690daeeb8a96da4a39f11fc27...,2017-03-15,0.400025,0,0.954861,0.023890,0.003140,<NA>,0.0,1.175081,...,-1,1,0,0.870115,0.141213953,1,0.008945,8,8,1
1699,0001812036f1558332e5c0880ecbad70b13a6f28ab04a8...,2017-03-27,0.410251,0,0.525142,0.018226,0.006648,1.607070804,0.0,0.266503,...,-1,0,0,0.000000,<NA>,0,0.005431,0,12,1
1700,0002d381bdd8048d76719042cf1eb63caf53b636f8aacd...,2017-03-19,1.007809,0,0.017698,0.816354,0.000443,0.345746458,0.0,0.007117,...,-1,0,0,0.000000,<NA>,0,0.003225,0,16,0


In [25]:
y_test = test['target'].values.get()
y_test.shape

(114729,)

In [26]:
from utils import amex_metric_np

In [27]:
amex_metric_np(y_test,yp[:,1])

np.float64(0.7979085206898009)

# Run triton inference on all the test data of 11 million samples!

In [28]:
test = cudf.read_parquet(f'{PATH}/test.parquet')
test['cid'], _ = test.customer_ID.factorize()
test['S_2'] = cudf.to_datetime(test['S_2'])

In [29]:
test.shape

(11363762, 191)

In [30]:
cpu_workers = 16
test_ds = TestRnnDataset(test,config)
test_dl = DataLoader(test_ds, batch_size=batch_size,
                    shuffle=False, num_workers=cpu_workers,
                    drop_last=False)

RnnDataset not used columns:
['customer_ID', 'cid', 'S_2']


In [31]:
%%time

yps = []
for x,xo in test_dl:
    rnn_fea = triton_predict('AutoRegressiveRNN',x.numpy())
    x = np.hstack([xo[:,-1,:],rnn_fea[:,-1,:]])
    pred = triton_predict('amex_xgb',x)
    yps.append(pred)
yp = np.vstack(yps)

CPU times: user 8.09 s, sys: 3.77 s, total: 11.9 s
Wall time: 1min 1s
